# Fine-tune DistilBERT — Essay Scorer (Stage 3 / Transformer Upgrade)

This notebook fine-tunes `distilbert-base-uncased` to predict essay scores (0–100),
as a drop-in upgrade for the Scikit-learn baseline model in `backend/app.py`.

**Before running:** In Colab, go to `Runtime > Change runtime type > T4 GPU` (free tier).
Training on CPU will be extremely slow — make sure the GPU is actually selected.

**Time estimate:** ~20–40 minutes on a free T4 GPU for ~1,800 essays, 3 epochs.

**What this notebook does:**
1. Clones your GitHub repo (to reuse the same dataset CSV you already trained the baseline on)
2. Tokenizes essays with DistilBERT's tokenizer
3. Fine-tunes DistilBERT with a regression head (1 output = predicted score)
4. Evaluates Mean Absolute Error, so you can directly compare it to the baseline's 6.49
5. Saves the fine-tuned model — download it and drop it into `backend/transformer_model/`

## 1. Install dependencies

In [1]:
%pip uninstall -y torchvision torchaudio
%pip install -q transformers datasets accelerate scikit-learn

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


## 2. Get the dataset

**Option A (recommended): clone your GitHub repo.** Replace the URL below with
your actual repo URL — since you've already pushed the project, this pulls in
`backend/sample_data/asap_set1_rescaled.csv` directly, so you're training on
the exact same data as the baseline model. That makes the MAE comparison fair.

**Option B: upload manually.** If you'd rather not clone the repo, comment out
the clone cell and instead click the folder icon on the left sidebar in Colab
and drag `asap_set1_rescaled.csv` in directly, then update `CSV_PATH` below.

In [11]:
# --- Option A: clone your repo ---
REPO_URL = "https://github.com/Al7bh/Email-Analyzer.git"  # <-- replace this

!git clone $REPO_URL project

CSV_PATH = "project/backend/sample_data/asap_set1_rescaled.csv"

# --- Option B: if you uploaded the CSV manually instead, use this path ---
#CSV_PATH = "backend\\sample_data\\asap_set1_rescaled.csv"

fatal: destination path 'project' already exists and is not an empty directory.


In [3]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} essays")
print(df["score"].describe())
df.head(3)

Loaded 1783 essays
count    1783.000000
mean       65.283231
std        15.385652
min         0.000000
25%        60.000000
50%        60.000000
75%        80.000000
max       100.000000
Name: score, dtype: float64


,essay,score
0,"Dear local newspaper, I think effects computer...",60.0
1,"Dear @CAPS1 @CAPS2, I believe that using compu...",70.0
2,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",50.0


## 3. Train / validation split

Same 80/20 split style as `train.py` uses for the baseline, so the two models'
reported MAE numbers are comparable in your report.

In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)}  |  Validation: {len(val_df)}")

Train: 1426  |  Validation: 357


## 4. Tokenize

DistilBERT has a 512-token limit. Essays get truncated if longer — for the
ASAP set-1 essays (mostly a few hundred words) this rarely matters, but worth
knowing if you switch to a dataset with much longer essays later.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Scores are rescaled to 0-1 for training stability, then rescaled back to
# 0-100 at prediction time (see the predict cell near the end).
train_df = train_df.copy()
val_df = val_df.copy()
train_df["label"] = train_df["score"] / 100.0
val_df["label"] = val_df["score"] / 100.0

train_ds = Dataset.from_pandas(train_df[["essay", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["essay", "label"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["essay"], truncation=True, padding="max_length", max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1426 [00:00<?, ? examples/s]

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

## 5. Load the model with a regression head

`num_labels=1` + `problem_type="regression"` turns DistilBERT's classification
head into a single continuous output instead of class probabilities — exactly
what we want for a 0–1 (later rescaled to 0–100) score prediction.

In [6]:
from transformers import AutoModelForSequenceClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > T4 GPU, then re-run from the top.")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
).to(device)

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Train

In [7]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import mean_absolute_error

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    # Rescale both back to 0-100 so the reported MAE is directly comparable
    # to the baseline model's 6.49.
    mae = mean_absolute_error(labels * 100, preds * 100)
    return {"mae_0_100": mae}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae_0_100",
    greater_is_better=False,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Mae 0 100
1,0.010434,0.008963,7.753637
2,0.006524,0.005655,5.976863
3,0.005933,0.005452,5.852947


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=537, training_loss=0.011859867146006288, metrics={'train_runtime': 240.4401, 'train_samples_per_second': 17.792, 'train_steps_per_second': 2.233, 'total_flos': 566685425240064.0, 'train_loss': 0.011859867146006288, 'epoch': 3.0})

## 7. Evaluate

Compare this MAE directly to the baseline's **6.49** (0–100 scale) from
`train.py`. If the transformer's MAE is meaningfully lower, that's your
quantitative evidence for the upgrade in your report.

In [8]:
metrics = trainer.evaluate()
print(metrics)
print(f"\nTransformer MAE: {metrics['eval_mae_0_100']:.2f}  (baseline was 6.49)")

Training Loss,Validation Loss,Epoch,Mae 0 100
0.005933,0.005452,3,5.852947


{'eval_loss': 0.005451752804219723, 'eval_mae_0_100': 5.852947235107422}

Transformer MAE: 5.85  (baseline was 6.49)


## 8. Save and download the model

This saves the fine-tuned model + tokenizer, zips them, and downloads the zip
to your computer. Unzip it into `backend/transformer_model/` in your project.

In [9]:
SAVE_DIR = "essay_scorer_distilbert"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -r essay_scorer_distilbert.zip {SAVE_DIR}

from google.colab import files
files.download("essay_scorer_distilbert.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: essay_scorer_distilbert/ (stored 0%)
  adding: essay_scorer_distilbert/config.json (deflated 49%)
  adding: essay_scorer_distilbert/tokenizer.json (deflated 71%)
  adding: essay_scorer_distilbert/model.safetensors (deflated 8%)
  adding: essay_scorer_distilbert/training_args.bin (deflated 54%)
  adding: essay_scorer_distilbert/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Quick sanity check before downloading

Try the fine-tuned model on a couple of essays directly in Colab, so you know
it's producing sane scores before you wire it into the Flask backend.

In [10]:
def predict_score(essay_text):
    inputs = tokenizer(essay_text, truncation=True, padding="max_length", max_length=512, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        output = model(**inputs)
    raw = output.logits.item()
    score = max(0, min(100, round(raw * 100)))
    return score

strong_essay = """Technology has changed how people communicate in the modern world.
In the past, people wrote letters and waited weeks for a reply. Today, messages
arrive instantly through phones and computers. This shift has made communication
faster, but it has also reduced the depth of many conversations we have with each other."""

weak_essay = """i think social media is bad and good. some people like it some people
dont. it can help you talk to friends but also it can waste time. i dont know what
to say more about this topic because its hard."""

print("Strong essay score:", predict_score(strong_essay))
print("Weak essay score:", predict_score(weak_essay))

Strong essay score: 27
Weak essay score: 10
